# 数据读取：CSV / Excel / JSON / SQL 全搞定

> 这是「数据分析从入门到精通」系列的第 8 篇。DataFrame 会用了，数据从哪来？这篇把最常见的几种数据格式的读取方式一次讲清楚，顺带教你几个超实用的读取技巧。

---

嗨，我是小荷。

上一篇我们手动造了好多 DataFrame，但真实工作里数据不是你造的——是产品经理给你发来的 Excel，是数据库里导出的 CSV，是第三方接口返回的 JSON……

这让我想到萧何当初接手咸阳的时候，第一件事不是扩张，而是**把秦朝留下来的户籍档案、地图、官府文书全部收集起来**。信息就是力量，但前提是你得**先能把数据读进来**。

今天就解决这个问题。

---

## 读取 CSV 文件

CSV 是最常见的数据格式，`pd.read_csv()` 是你用得最多的函数。

### 基础用法

In [ ]:
import pandas as pd

# 最简单的读法
df = pd.read_csv('data.csv')

# 指定编码（中文文件常用）
df = pd.read_csv('data.csv', encoding='utf-8')
df = pd.read_csv('data.csv', encoding='gbk')  # Windows 导出的中文文件

# 指定分隔符（默认是逗号）
df = pd.read_csv('data.tsv', sep='\t')        # tab 分隔
df = pd.read_csv('data.csv', sep=';')         # 分号分隔


### 常用参数详解

In [ ]:
df = pd.read_csv(
    'orders.csv',
    encoding='utf-8',
    index_col='订单ID',      # 指定某列作为行索引
    usecols=['用户ID', '金额', '商品类别'],   # 只读取指定列，节省内存
    nrows=1000,              # 只读前1000行（调试大文件时超好用）
    skiprows=[1, 2],         # 跳过指定行
    na_values=['N/A', '缺失', '-'],  # 指定哪些值算缺失值
    parse_dates=['下单时间'],  # 自动解析为日期类型
    dtype={'用户ID': str}    # 指定列的数据类型（避免数字ID被当成int）
)


> 💡 **实战技巧**：大文件调试时先用 `nrows=100` 读一小部分，确认格式没问题再读全量。别上来就 `read_csv` 一个 5GB 的文件，内存可能撑不住哈哈。

### 写出 CSV

In [ ]:
df.to_csv('output.csv', index=False, encoding='utf-8-sig')
# index=False：不写出行索引
# utf-8-sig：Windows Excel 打开中文不乱码


---

## 读取 Excel 文件

In [ ]:
# 需要安装 openpyxl：pip install openpyxl

# 读取默认第一个 sheet
df = pd.read_excel('report.xlsx')

# 读取指定 sheet
df = pd.read_excel('report.xlsx', sheet_name='销售数据')
df = pd.read_excel('report.xlsx', sheet_name=1)  # 按位置（从0开始）

# 一次性读取所有 sheet
all_sheets = pd.read_excel('report.xlsx', sheet_name=None)
# all_sheets 是个字典：{'sheet1': df1, 'sheet2': df2}
for name, df in all_sheets.items():
    print(f"{name}: {df.shape}")


### 写出 Excel

In [ ]:
# 写出单个 sheet
df.to_excel('output.xlsx', sheet_name='分析结果', index=False)

# 写出多个 sheet
with pd.ExcelWriter('output.xlsx') as writer:
    df1.to_excel(writer, sheet_name='原始数据', index=False)
    df2.to_excel(writer, sheet_name='汇总结果', index=False)


---

## 读取 JSON 文件

In [ ]:
import json

# 方式1：直接用 Pandas 读（适合标准格式）
df = pd.read_json('data.json')

# 方式2：先用 json 库解析，再转 DataFrame（适合复杂嵌套格式）
with open('data.json', 'r', encoding='utf-8') as f:
    raw = json.load(f)

df = pd.DataFrame(raw)

# 读取 API 接口返回的 JSON 数据
import requests
response = requests.get('https://api.example.com/data')
df = pd.DataFrame(response.json())


### 嵌套 JSON 的处理

In [ ]:
# 嵌套结构
data = [
    {"user": "张三", "orders": {"count": 5, "total": 1200}},
    {"user": "李四", "orders": {"count": 3, "total": 800}},
]

df = pd.json_normalize(data)  # 自动展开嵌套结构
print(df)
#    user  orders.count  orders.total
# 0  张三             5          1200
# 1  李四             3           800


---

## 读取 SQL 数据库

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# 创建数据库连接（以 MySQL 为例）
engine = create_engine('mysql+pymysql://用户名:密码@host:3306/数据库名')

# 读取整张表
df = pd.read_sql_table('orders', con=engine)

# 执行 SQL 查询
sql = """
    SELECT user_id, SUM(amount) as total_amount
    FROM orders
    WHERE order_date >= '2024-01-01'
    GROUP BY user_id
    ORDER BY total_amount DESC
    LIMIT 100
"""
df = pd.read_sql(sql, con=engine)

# 写入数据库
df.to_sql('result_table', con=engine, if_exists='replace', index=False)


> 💡 如果只是练习，推荐用 SQLite（不需要安装服务器）：
>

In [ ]:
> import sqlite3
> conn = sqlite3.connect('local.db')
> df = pd.read_sql('SELECT * FROM orders', con=conn)
> 


---

## 实战：读取一份真实电商数据

用项目里的全国城市数据做演示：

In [3]:
import pandas as pd

# 读取 CSV
df = pd.read_csv('全国城市数据.csv', encoding='gbk')

# 快速体检
print(f"数据规模: {df.shape[0]} 行 × {df.shape[1]} 列")
print("\n列名和类型:")
print(df.dtypes)
print("\n前5行:")
print(df.head())
print("\n基本统计:")
print(df.describe().round(2))

# 常见问题检查
print(f"\n缺失值:\n{df.isnull().sum()}")
print(f"\n重复行: {df.duplicated().sum()} 行")


数据规模: 343 行 × 8 列

列名和类型:
省份                  str
城市名称                str
城市级别                str
常住人口(万人)          int64
面积(km2)           int64
人均面积(km2/万人)    float64
火星纬度(GCJ02)     float64
火星经度(GCJ02)     float64
dtype: object

前5行:
       省份     城市名称 城市级别  常住人口(万人)  面积(km2)  人均面积(km2/万人)  火星纬度(GCJ02)  \
0  内蒙古自治区    呼伦贝尔市   二线       248   252777   1019.262097      49.2126   
1  内蒙古自治区    鄂尔多斯市   二线       215    86752    403.497674      39.6081   
2  内蒙古自治区      赤峰市   二线       430    90021    209.351163      42.2566   
3  内蒙古自治区      通辽市   二线       312    59535    190.817308      43.6529   
4     四川省  凉山彝族自治州   二线       484    60423    124.840909      27.8868   

   火星经度(GCJ02)  
0     119.7654  
1     110.0235  
2     118.9220  
3     122.2440  
4     102.2674  

基本统计:
       常住人口(万人)    面积(km2)  人均面积(km2/万人)  火星纬度(GCJ02)  火星经度(GCJ02)
count    343.00     343.00        343.00       343.00       343.00
mean     421.22   28313.26        341.73        33.18       111.34
std      378.

---

## 常见问题排查

| 问题 | 原因 | 解决方法 |
|------|------|---------|
| 中文乱码 | 编码不对 | 试试 `encoding='gbk'` 或 `utf-8-sig` |
| 数字列读成字符串 | 有逗号或空格 | `dtype=float` 或先清洗再读 |
| 日期列是字符串 | 未解析 | `parse_dates=['日期列']` |
| 内存溢出 | 文件太大 | `nrows` 分块读取或用 `chunksize` |
| Excel 读取报错 | 缺少依赖 | `pip install openpyxl xlrd` |

---

## 本篇小结

| 格式 | 读取函数 | 写出函数 |
|------|---------|---------|
| CSV | `pd.read_csv()` | `df.to_csv()` |
| Excel | `pd.read_excel()` | `df.to_excel()` |
| JSON | `pd.read_json()` | `df.to_json()` |
| SQL | `pd.read_sql()` | `df.to_sql()` |

**一个好习惯**：读完数据立刻跑 `df.info()` + `df.isnull().sum()`，先摸清楚数据的"家底"，再开始分析。

---

## 课后练习

1. 找一份你手边的 CSV 或 Excel 文件，用 Pandas 读进来
2. 用 `df.info()` 和 `df.describe()` 做一次"数据体检"
3. 尝试把某几列导出为新的 CSV 文件

没有数据？可以用项目里的 `全国城市数据.csv` 练手。
这几个任务比较简单，我就不贴代码了。

评论区告诉我你读了什么数据，遇到什么坑 👀

本篇完整代码包括练习题解答都已经上传至 GitHub 仓库，欢迎 Clone。

---

## 下期预告

> **第 9 篇：数据查看与描述性统计**
>
> 数据读进来了，怎么快速了解它的全貌？一行代码 `df.describe()` 背后有多少秘密？下篇带你把数据"看透"。

---

👇 点「在看」，推给身边搞数据的朋友  
💬 评论区说说你遇到过最奇葩的数据格式  
⭐ 关注公众号，Pandas 系列持续更新

---

*「数据分析从入门到精通」系列 · 第 8 篇*  
*上一篇：[第 7 篇：Pandas 入门 — Series 与 DataFrame]*  
*下一篇：第 9 篇：数据查看与描述性统计*